In [6]:
#!/usr/bin/env python
"""
Score genes for pain / nociception / DRG literature support via PubMed.
Uses server-side PubMed boolean queries (counts only), so it's fast.

Tier 1 (strongest): gene in TITLE         + strong term in title/abstract   (3 pts)
Tier 2:             gene in ABSTRACT only + strong term in title/abstract   (2 pts)
Tier 3 (weakest):   gene in title/abstract + weak term, but NO strong term  (1 pt)

Saves up to MAX_PMIDS PMIDs per gene (strongest tier first), not per tier.
best_tier = 0 means no papers matched any tier.
"""
import argparse
import re
import time
import pandas as pd
from Bio import Entrez

STRONG = ["pain", "nociception", "nociceptive", 
          "dorsal root ganglia", "dorsal root ganglion", "nociceptor", 
          "neuropathic", "neuropathy", "allodynia", "hyperalgesia", "analgesia"]
WEAK = ["hurt", "injury", "spinal cord", "peripheral nerve",
        "sensory processing", "sensory neuron",
        "central nervous system", "peripheral nervous system"]

# Optional extra names (checked in title/abstract alongside the symbol).
# Be careful with short/ambiguous aliases -- they add false positives.
ALIASES = {
    "CEMIP":   ["KIAA1199"],
    "BCL11B":  ["CTIP2"],
    "NSG1":    ["NEEP21"],
    "SPINT1":  ["HAI-1"],
    "COL25A1": ["CLAC-P"],
    "VSNL1":   ["VILIP-1"],
}

MAX_PMIDS = 50


def or_block(terms, field):
    return "(" + " OR ".join(f'"{t}"[{field}]' for t in terms) + ")"


def build_queries(gene, aliases=()):
    names = [gene] + [a for a in aliases if a and a != gene]
    g_title = or_block(names, "Title")
    g_tiab = or_block(names, "Title/Abstract")
    strong = or_block(STRONG, "Title/Abstract")
    weak = or_block(WEAK, "Title/Abstract")
    return {
        "tier1": f"({g_title} AND {strong})",
        "tier2": f"(({g_tiab} NOT {g_title}) AND {strong})",
        "tier3": f"(({g_tiab} AND {weak}) NOT {strong})",
        "total": g_tiab,
    }


def esearch(query, retmax=0, tries=5):
    for attempt in range(tries):
        try:
            h = Entrez.esearch(db="pubmed", term=query, retmax=retmax,
                               sort="relevance")
            rec = Entrez.read(h)
            h.close()
            return int(rec["Count"]), rec["IdList"]
        except Exception as e:
            print(f"  retry {attempt + 1} after error: {e}")
            time.sleep(2 ** attempt)
    return None, []


def score_gene(gene, aliases=(), max_pmids=MAX_PMIDS, delay=0.4):
    q = build_queries(gene, aliases)
    out = {"gene": gene, "aliases_used": ";".join(aliases)}

    top_pmids, top_tiers = [], []
    for tier_num, tier in [(1, "tier1"), (2, "tier2"), (3, "tier3")]:
        remaining = max_pmids - len(top_pmids)
        # retmax=0 still returns the count, just no IDs once we're full
        n, ids = esearch(q[tier], retmax=max(remaining, 0))
        out[f"n_{tier}"] = n
        for pid in ids:
            if pid not in top_pmids and len(top_pmids) < max_pmids:
                top_pmids.append(pid)
                top_tiers.append(str(tier_num))
        time.sleep(delay)

    out["top_pmids"] = ";".join(top_pmids)
    out["top_pmid_tiers"] = ";".join(top_tiers)

    n, _ = esearch(q["total"], retmax=0)
    out["n_gene_total"] = n
    time.sleep(delay)

    n1, n2, n3 = (out["n_tier1"], out["n_tier2"], out["n_tier3"])
    if None in (n1, n2, n3):
        out["best_tier"], out["weighted_score"] = None, None
    else:
        out["best_tier"] = 1 if n1 else 2 if n2 else 3 if n3 else 0
        out["weighted_score"] = 3 * n1 + 2 * n2 + n3
    return out


def run(email, infile="gene_evidence.csv", outfile="gene_pubmed_scores.csv",
        merged_out="gene_evidence_with_pubmed.csv", tier="B",
        gene_col="gene", tier_col="network_tier", alias_col="aliases",
        api_key=None, max_pmids=MAX_PMIDS):
    Entrez.email = email
    if api_key:
        Entrez.api_key = api_key
    delay = 0.12 if api_key else 0.4   # NCBI: 10 req/s with key, 3/s without

    df = pd.read_csv(infile, sep=None, engine="python")  # sniffs tab/comma
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
    sel = df if tier.lower() == "all" else df[df[tier_col].astype(str) == tier]
    print(f"Scoring {len(sel)} genes (tier filter: {tier})")

    rows = []
    for i, (_, r) in enumerate(sel.iterrows(), 1):
        gene = str(r[gene_col]).strip()
        al = list(ALIASES.get(gene, []))
        if alias_col in sel.columns and pd.notna(r.get(alias_col)):
            al += [a.strip() for a in re.split(r"[;,]", str(r[alias_col])) if a.strip()]
        print(f"[{i}/{len(sel)}] {gene}")
        rows.append(score_gene(gene, al, max_pmids=max_pmids, delay=delay))

    res = pd.DataFrame(rows).sort_values("weighted_score", ascending=False)
    res.to_csv(outfile, index=False)
    merged = df.merge(res.add_prefix("pm_").rename(columns={"pm_gene": gene_col}),
                      on=gene_col, how="left")
    merged.to_csv(merged_out, index=False)
    print(f"Saved {outfile} and {merged_out}")
    return res


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--email", required=True)
    ap.add_argument("--api-key", default=None)
    ap.add_argument("--infile", default="gene_evidence.csv")
    ap.add_argument("--outfile", default="gene_pubmed_scores.csv")
    ap.add_argument("--merged-out", default="gene_evidence_with_pubmed.csv")
    ap.add_argument("--tier", default="B", help="network_tier to score, or 'all'")
    ap.add_argument("--max-pmids", type=int, default=MAX_PMIDS)
    args = ap.parse_args()
    run(args.email, args.infile, args.outfile, args.merged_out,
        args.tier, api_key=args.api_key, max_pmids=args.max_pmids)

usage: ipykernel_launcher.py [-h] --email EMAIL [--api-key API_KEY]
                             [--infile INFILE] [--outfile OUTFILE]
                             [--merged-out MERGED_OUT] [--tier TIER]
                             [--max-pmids MAX_PMIDS]
ipykernel_launcher.py: error: the following arguments are required: --email


SystemExit: 2

/lisc/data/scratch/menche/ines/general_env/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [7]:
res = run(email="gerardursi99@univie.ac.at", infile="gene_evidence.csv", tier="all")
res.head(20)

Scoring 82 genes (tier filter: all)
[1/82] PLXND1
[2/82] ALKAL2
[3/82] ASIC1
[4/82] DAGLA
[5/82] FXYD6
[6/82] GFRA1
[7/82] KCND3
[8/82] NPTX2
[9/82] NTNG1
[10/82] NTRK3
[11/82] TMEM45B
[12/82] CHRNA3
[13/82] DAPK1
[14/82] GPM6A
[15/82] SYT2
[16/82] LITAF
[17/82] PLXNA2
[18/82] SCN3B
[19/82] SLC7A14
[20/82] UGT8
[21/82] VSNL1
[22/82] CACNG5
[23/82] CD9
[24/82] CREB3L1
[25/82] GPRC5C
[26/82] HS6ST2
[27/82] ISL2
[28/82] RAMP3
[29/82] SCGN
[30/82] SDC3
[31/82] SERPINA1
[32/82] AHNAK
[33/82] AMIGO2
[34/82] AMOTL1
[35/82] ARHGAP18
[36/82] ASPHD2
[37/82] BAIAP2L1
[38/82] BCL11B
[39/82] BRINP2
[40/82] BTN2A2
[41/82] C11ORF87
[42/82] CDH1
[43/82] CEMIP
[44/82] COL25A1
[45/82] CPNE5
[46/82] DDR2
[47/82] EMP1
[48/82] FHL2
[49/82] FSTL4
[50/82] HPCA
[51/82] HSD17B13
[52/82] HSPB6
[53/82] ID4
[54/82] IGFBP6
[55/82] IGSF3
[56/82] LDB2
[57/82] LNX1
[58/82] LY6H
[59/82] MDGA1
[60/82] MPPED1
[61/82] MYO1B
[62/82] NECTIN1
[63/82] NELL1
[64/82] NPNT
[65/82] NSG1
[66/82] PKN1
[67/82] PLK2
[68/82] PTPDC1
[

,gene,aliases_used,n_tier1,n_tier2,n_tier3,top_pmids,top_pmid_tiers,n_gene_total,best_tier,weighted_score
2,ASIC1,,18,85,36,16002453;30124148;12509480;26589317;33314662;3...,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;2;2;2;2;2;...,306,1,260
22,CD9,,4,20,183,26857499;8219357;7529830;42800031;24038174;284...,1;1;1;1;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;...,3452,1,235
41,CDH1,,6,13,80,31367630;26364211;31785347;38651154;27184142;3...,1;1;1;1;1;1;2;2;2;2;2;2;2;2;2;2;2;2;2;3;3;3;3;...,4699,1,124
81,UST,,1,43,21,38159944;35162881;17344777;41059277;34040429;2...,1;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;...,1059,1,110
15,LITAF,,12,26,20,27927196;35608774;25342198;12525712;28211240;2...,1;1;1;1;1;1;1;1;1;1;1;1;2;2;2;2;2;2;2;2;2;2;2;...,342,1,108
9,NTRK3,,5,18,45,26061886;36740981;42754357;36924196;42724224;3...,1;1;1;1;1;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;...,1142,1,96
30,SERPINA1,,1,10,61,32768500;34580987;40281518;33788977;35332129;3...,1;2;2;2;2;2;2;2;2;2;2;3;3;3;3;3;3;3;3;3;3;3;3;...,1016,1,84
68,RBP4,,1,7,65,41943100;33155274;41852479;39076230;38812760;3...,1;2;2;2;2;2;2;2;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;...,1581,1,82
12,DAPK1,,0,1,77,32175411;23567086;40787892;30062675;24806680;2...,2;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;...,777,2,79
37,BCL11B,CTIP2,0,2,62,25756408;29423022;39173624;34440292;35685361;2...,2;2;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;3;...,730,2,66
